In [1]:
import gc
import torch
from diffusers import Flux2Pipeline


# ─────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────

LOCAL_BASE_DIR       = "./models/flux2-dev-bnb-4bit"   # contains text_encoder/, tokenizer/, transformer/, vae/, scheduler/
LOCAL_TURBO_DIR      = "./models/flux2-turbo-lora"
TURBO_LORA_FILENAME  = "flux.2-turbo-lora.safetensors"

DEVICE = "cuda:0"
DTYPE  = torch.bfloat16

OUTPUT_PATH = "dwarf_lord_by_AI.png"

TURBO_SIGMAS = [1.0, 0.6509, 0.4374, 0.2932, 0.1893, 0.1108, 0.0495, 0.00031]

HEIGHT = 1024
WIDTH  = 1024
GUIDANCE_SCALE      = 2.5
NUM_INFERENCE_STEPS = 8
SEED = 42


# ─────────────────────────────────────────────────────────────
#  PROMPT
# ─────────────────────────────────────────────────────────────

prompt = (
    "A close-up richly detailed oil-painted fantasy character portrait of a dwarf lord, "
    "in the style of classic epic fantasy book cover illustration, painted with visible "
    "brushwork, deep glazes, and masterful realistic rendering; dark warm background fading "
    "from soft tan light into deep umber shadow; stocky and powerfully built with a broad, "
    "muscular upper body and short bowed legs; mismatched eyes — one deep obsidian black, "
    "the other a vivid piercing green — painted with subtle wet reflective highlights; his "
    "nose is reduced to a rough stub, most of it sliced away in battle, a deep jagged scar "
    "running from cheekbone to jaw, rendered with realistic skin texture and healed tissue "
    "detail; his jaw is uneven and asymmetrical, short bristly mismatched-colored hair falls "
    "unevenly, catching warm rim light against the dark background; he looks directly at the "
    "viewer with a shrewd, knowing half-smile, head tilted slightly, radiating cunning and "
    "quiet authority; he wears fine dark burnished plate armor with intricate gold filigree "
    "engraving across the chestplate and pauldrons, a heavy crimson cape fastened by a gold "
    "lion-head clasp draped over one shoulder, the fabric rendered with deep realistic folds, "
    "soft velvet sheen, and directional highlights; one hand rests near the pommel of a sword "
    "at his hip, fine detail in the rings and knuckles; dramatic Rembrandt-style side lighting "
    "falls across his face and armor from camera-left, carving strong contrast between warm "
    "lit highlights and deep soft shadow; softly blurred moody background with subtle vignette, "
    "in the tradition of classic fantasy portrait painting, realistic skin texture with fine "
    "pores and natural asymmetry, muted rich color palette of crimson, aged gold, and dark "
    "steel, elegant and painterly, gallery-quality digital oil painting, sharp focus on the "
    "face, noble and commanding presence despite his stature"
)

# NOTE: Flux2Pipeline does not support negative prompts (see note above) —
# there is no negative_prompt parameter to use it with. Keeping this string
# here only for your own reference / for other tools that do support it.
negative_prompt_reference_only = (
    "flat cel-shaded animation, cartoon, anime style, vector art, chibi proportions, "
    "smooth plastic skin, glossy toon shading, thick black outlines, intact undamaged nose, "
    "symmetrical unscarred face, matching eye colors, blank vacant stare, looking away from "
    "camera, plain or drab clothing, timid expression, low detail, blurry, extra limbs, "
    "deformed hands, malformed face, text, watermark, signature, oversaturated flat colors, "
    "harsh overexposure, lens flare artifacts, unrealistic proportions, exaggerated scars "
    "beyond described, bright flat even lighting, washed-out colors, child-like features, "
    "tall average human proportions, non-dwarf body type, 3d render, video game render, "
    "airbrushed digital art"
)


# ─────────────────────────────────────────────────────────────
#  STEP 1 — Load ONLY text_encoder + tokenizer on GPU, encode prompts
# ─────────────────────────────────────────────────────────────

print(f"Loading text_encoder + tokenizer (only) from: {LOCAL_BASE_DIR}")

text_pipe = Flux2Pipeline.from_pretrained(
    LOCAL_BASE_DIR,
    transformer=None,     # skip — don't load the (large) transformer yet
    vae=None,              # skip — not needed for encoding
    torch_dtype=DTYPE,
    local_files_only=True,
).to(DEVICE)

print("Tokenizing + encoding prompt on GPU ...")
# NOTE: Flux2Pipeline has NO negative_prompt / negative_prompt_embeds support.
# It's a guidance-distilled model (like FLUX.1-dev) — guidance_scale is
# "embedded guidance" baked into the model, not classifier-free guidance
# against a negative prompt. encode_prompt() returns (prompt_embeds, text_ids);
# text_ids gets recomputed internally when we pass prompt_embeds to the main
# pipeline call, so we only need to keep prompt_embeds here.
with torch.no_grad():
    prompt_embeds, _text_ids = text_pipe.encode_prompt(
        prompt=prompt,
        device=DEVICE,
    )

# Move embeddings to CPU so they survive the text encoder being deleted
prompt_embeds = prompt_embeds.to("cpu")

# Free the text encoder from VRAM completely before loading the transformer
del text_pipe
gc.collect()
torch.cuda.empty_cache()
print("✅  Text encoder freed from VRAM.\n")

'housse of stark_chosen.mp3'

In [ ]:
print(f"Loading FLUX.2 [dev] transformer + VAE from: {LOCAL_BASE_DIR}")
pipe = Flux2Pipeline.from_pretrained(
    LOCAL_BASE_DIR,
    text_encoder=None,   # already encoded above — don't load it again
    tokenizer=None,
    torch_dtype=DTYPE,
    local_files_only=True,
).to(DEVICE)

print(f"Loading Turbo LoRA from: {LOCAL_TURBO_DIR}/{TURBO_LORA_FILENAME}")
pipe.load_lora_weights(LOCAL_TURBO_DIR, weight_name=TURBO_LORA_FILENAME)

print("✅  Pipeline ready.\n")

generate_kwargs = dict(
    height=HEIGHT,
    width=WIDTH,
    sigmas=TURBO_SIGMAS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=torch.Generator(DEVICE).manual_seed(SEED),
    num_images_per_prompt=1,
    prompt_embeds=prompt_embeds.to(DEVICE),
)

print("Generating image (8 turbo steps) ...")
image = pipe(**generate_kwargs).images[0]

image.save(OUTPUT_PATH)
print(f"\n✅  Saved: {OUTPUT_PATH}")


# ─────────────────────────────────────────────────────────────
#  CLEANUP
# ─────────────────────────────────────────────────────────────

pipe.to("cpu")
del pipe
gc.collect()
torch.cuda.empty_cache()